In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ============================================================
# Cell 1: Imports, random seed, device, and global settings
# ============================================================

from pathlib import Path
from typing import Dict, List, Tuple
import ast
import gc
import json
import math
import os
import random
import re
import time
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoTokenizer,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.width", 200)

SEED = 42


def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducible evaluation."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # These settings improve reproducibility.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Environment information")
print("=" * 70)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
            2,
        ),
        "GB",
    )


Environment information
PyTorch version: 2.10.0+cpu
CUDA available: False
Selected device: cpu


In [2]:
# ============================================================
# Cell 2: Paths and evaluation configuration
# ============================================================

# Original factual probing dataset
PROBE_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_dataset.csv"
)

REJECTED_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_rejected.csv"
)

STATS_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_stats.csv"
)

# Base model used for comparison
XLMR_BASE_MODEL = "xlm-roberta-base"

# Fine-tuned XLM-R-KG model
XLMR_KG_MODEL_PATH = Path(
    "/kaggle/input/notebooks/sammir79/"
    "finetunning-xlm-roberta/xlm_roberta_kg_mlm"
)

# Output directory
OUTPUT_DIR = Path("/kaggle/working/factual_probe_xlm_roberta")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation settings
BATCH_SIZE = 32
MAX_LENGTH = 128
TOP_K_TO_SAVE = 10

# Use mixed precision on GPU to reduce memory consumption.
USE_AMP = torch.cuda.is_available()

print("=" * 70)
print("Paths and configuration")
print("=" * 70)

print(f"Probe file:\n  {PROBE_FILE}")
print(f"Probe file exists: {PROBE_FILE.exists()}")
print()

print(f"XLM-R base model:\n  {XLMR_BASE_MODEL}")
print()

print(f"XLM-R-KG path:\n  {XLMR_KG_MODEL_PATH}")
print(f"XLM-R-KG path exists: {XLMR_KG_MODEL_PATH.exists()}")
print()

print(f"Output directory:\n  {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Maximum sequence length: {MAX_LENGTH}")
print(f"Saved top-k predictions: {TOP_K_TO_SAVE}")
print(f"Automatic mixed precision: {USE_AMP}")

if not PROBE_FILE.exists():
    raise FileNotFoundError(
        f"Probe dataset was not found:\n{PROBE_FILE}"
    )

if not XLMR_KG_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Fine-tuned XLM-R-KG model was not found:\n"
        f"{XLMR_KG_MODEL_PATH}"
    )


Paths and configuration
Probe file:
  /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_dataset.csv
Probe file exists: True

XLM-R base model:
  xlm-roberta-base

XLM-R-KG path:
  /kaggle/input/notebooks/sammir79/finetunning-xlm-roberta/xlm_roberta_kg_mlm
XLM-R-KG path exists: True

Output directory:
  /kaggle/working/factual_probe_xlm_roberta
Batch size: 32
Maximum sequence length: 128
Saved top-k predictions: 10
Automatic mixed precision: False


In [3]:
# ============================================================
# Cell 3: Inspect the fine-tuned model directory
# ============================================================

kg_model_files = sorted(
    [
        str(path.relative_to(XLMR_KG_MODEL_PATH))
        for path in XLMR_KG_MODEL_PATH.rglob("*")
        if path.is_file()
    ]
)

print(f"Number of files under KG model path: {len(kg_model_files)}")

for file_name in kg_model_files[:100]:
    print(file_name)

required_model_files = [
    "config.json",
    "model.safetensors",
]

for file_name in required_model_files:
    file_path = XLMR_KG_MODEL_PATH / file_name
    print(f"{file_name}: {file_path.exists()}")

if not (XLMR_KG_MODEL_PATH / "config.json").exists():
    raise FileNotFoundError(
        "config.json is missing from the XLM-R-KG model directory."
    )

if not (
    (XLMR_KG_MODEL_PATH / "model.safetensors").exists()
    or (XLMR_KG_MODEL_PATH / "pytorch_model.bin").exists()
):
    raise FileNotFoundError(
        "Neither model.safetensors nor pytorch_model.bin was found."
    )


Number of files under KG model path: 28
all_results.json
checkpoint-12000/config.json
checkpoint-12000/model.safetensors
checkpoint-12000/optimizer.pt
checkpoint-12000/rng_state.pth
checkpoint-12000/scaler.pt
checkpoint-12000/scheduler.pt
checkpoint-12000/tokenizer.json
checkpoint-12000/tokenizer_config.json
checkpoint-12000/trainer_state.json
checkpoint-12000/training_args.bin
checkpoint-12500/config.json
checkpoint-12500/model.safetensors
checkpoint-12500/optimizer.pt
checkpoint-12500/rng_state.pth
checkpoint-12500/scaler.pt
checkpoint-12500/scheduler.pt
checkpoint-12500/tokenizer.json
checkpoint-12500/tokenizer_config.json
checkpoint-12500/trainer_state.json
checkpoint-12500/training_args.bin
config.json
model.safetensors
tokenizer.json
tokenizer_config.json
train_results.json
trainer_state.json
training_args.bin
config.json: True
model.safetensors: True


In [4]:
# ============================================================
# Cell 4: Load and validate the factual probing dataset
# ============================================================

probe_df = pd.read_csv(
    PROBE_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

print("=" * 70)
print("Original dataset")
print("=" * 70)
print(f"Rows: {len(probe_df):,}")
print(f"Columns: {len(probe_df.columns)}")
print("\nColumn names:")
print(probe_df.columns.tolist())

required_columns = [
    "fact_id",
    "subject",
    "predicate",
    "object",
    "prompt",
    "gold_answer",
    "valid",
    "object_token_count",
]

missing_columns = [
    column
    for column in required_columns
    if column not in probe_df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )

display(probe_df.head(5))


Original dataset
Rows: 1,206
Columns: 15

Column names:
['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN


In [5]:
# ============================================================
# Cell 5: Basic dataset cleaning and original validity filters
# ============================================================

def normalize_boolean_series(series: pd.Series) -> pd.Series:
    """Convert common boolean representations to Python booleans."""
    true_values = {
        "true",
        "1",
        "yes",
        "y",
        "t",
    }

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(true_values)
    )


evaluation_df = probe_df.copy()

# Preserve the original row index for traceability.
evaluation_df["original_row_index"] = evaluation_df.index

# Normalize text columns.
evaluation_df["prompt"] = (
    evaluation_df["prompt"]
    .fillna("")
    .astype(str)
    .str.strip()
)

evaluation_df["gold_answer"] = (
    evaluation_df["gold_answer"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Convert the validity field safely.
evaluation_df["normalized_valid"] = normalize_boolean_series(
    evaluation_df["valid"]
)

# Convert object_token_count to numeric.
evaluation_df["object_token_count_numeric"] = pd.to_numeric(
    evaluation_df["object_token_count"],
    errors="coerce",
)

initial_rows = len(evaluation_df)

# Apply the original dataset validity constraints.
evaluation_df = evaluation_df[
    evaluation_df["normalized_valid"]
].copy()

after_valid = len(evaluation_df)

evaluation_df = evaluation_df[
    evaluation_df["object_token_count_numeric"] == 1
].copy()

after_original_single_token = len(evaluation_df)

evaluation_df = evaluation_df[
    evaluation_df["prompt"].ne("")
    & evaluation_df["gold_answer"].ne("")
].copy()

after_non_empty = len(evaluation_df)

# Require exactly one literal [MASK] in the original prompt.
evaluation_df["original_mask_count"] = (
    evaluation_df["prompt"]
    .str.count(r"\[MASK\]")
)

invalid_mask_rows_df = evaluation_df[
    evaluation_df["original_mask_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["original_mask_count"] == 1
].copy()

after_mask_filter = len(evaluation_df)

print("=" * 70)
print("Initial filtering report")
print("=" * 70)
print(f"Original rows:                         {initial_rows:,}")
print(f"After valid=True:                     {after_valid:,}")
print(
    f"After original object_token_count=1: {after_original_single_token:,}"
)
print(f"After non-empty text filtering:       {after_non_empty:,}")
print(f"After exactly one [MASK]:             {after_mask_filter:,}")
print(
    f"Removed due to invalid mask count:    "
    f"{len(invalid_mask_rows_df):,}"
)

if len(evaluation_df) == 0:
    raise ValueError(
        "No rows remained after the initial dataset filters."
    )

print("\nPredicate distribution after initial filtering:")
display(
    evaluation_df["predicate"]
    .value_counts(dropna=False)
    .rename_axis("predicate")
    .reset_index(name="count")
)


Initial filtering report
Original rows:                         1,206
After valid=True:                     1,206
After original object_token_count=1: 1,206
After non-empty text filtering:       1,206
After exactly one [MASK]:             1,206
Removed due to invalid mask count:    0

Predicate distribution after initial filtering:


,predicate,count
0,ملیت,242
1,زبان,240
2,محل تولد,238
3,استان,233
4,کشور,208
5,زبان رسمی,45


In [6]:
# ============================================================
# Cell 6: Load and verify XLM-R tokenizers
# ============================================================

print("Loading the baseline XLM-R tokenizer...")

base_tokenizer = AutoTokenizer.from_pretrained(
    XLMR_BASE_MODEL,
    use_fast=True,
)

print("Loading the XLM-R-KG tokenizer...")

kg_tokenizer = AutoTokenizer.from_pretrained(
    str(XLMR_KG_MODEL_PATH),
    use_fast=True,
)

print("\n" + "=" * 70)
print("Tokenizer information")
print("=" * 70)

print("Baseline tokenizer:")
print(f"  Name/path: {base_tokenizer.name_or_path}")
print(f"  Vocabulary length: {len(base_tokenizer):,}")
print(f"  Mask token: {base_tokenizer.mask_token}")
print(f"  Mask token ID: {base_tokenizer.mask_token_id}")
print(f"  PAD token: {base_tokenizer.pad_token}")
print(f"  PAD token ID: {base_tokenizer.pad_token_id}")

print("\nKG tokenizer:")
print(f"  Name/path: {kg_tokenizer.name_or_path}")
print(f"  Vocabulary length: {len(kg_tokenizer):,}")
print(f"  Mask token: {kg_tokenizer.mask_token}")
print(f"  Mask token ID: {kg_tokenizer.mask_token_id}")
print(f"  PAD token: {kg_tokenizer.pad_token}")
print(f"  PAD token ID: {kg_tokenizer.pad_token_id}")

if base_tokenizer.mask_token is None:
    raise ValueError(
        "The baseline tokenizer has no mask token."
    )

if kg_tokenizer.mask_token is None:
    raise ValueError(
        "The KG tokenizer has no mask token."
    )

# Check basic compatibility.
if len(base_tokenizer) != len(kg_tokenizer):
    raise ValueError(
        "Baseline and KG tokenizers have different vocabulary sizes:\n"
        f"Baseline: {len(base_tokenizer)}\n"
        f"KG: {len(kg_tokenizer)}"
    )

if base_tokenizer.mask_token_id != kg_tokenizer.mask_token_id:
    raise ValueError(
        "Baseline and KG tokenizers have different mask token IDs:\n"
        f"Baseline: {base_tokenizer.mask_token_id}\n"
        f"KG: {kg_tokenizer.mask_token_id}"
    )

# Compare the complete token-to-ID mappings.
base_vocab = base_tokenizer.get_vocab()
kg_vocab = kg_tokenizer.get_vocab()

vocabularies_identical = base_vocab == kg_vocab

print(
    "\nComplete vocabulary mappings identical:",
    vocabularies_identical,
)

if not vocabularies_identical:
    raise ValueError(
        "The complete baseline and KG tokenizer vocabularies are not "
        "identical. The same verified gold token IDs therefore cannot "
        "safely be used for both models."
    )

# Use one verified tokenizer for both models.
tokenizer = base_tokenizer

print("\nTokenizer compatibility check passed successfully.")


Loading the baseline XLM-R tokenizer...
Loading the XLM-R-KG tokenizer...

Tokenizer information
Baseline tokenizer:
  Name/path: xlm-roberta-base
  Vocabulary length: 250,002
  Mask token: <mask>
  Mask token ID: 250001
  PAD token: <pad>
  PAD token ID: 1

KG tokenizer:
  Name/path: /kaggle/input/notebooks/sammir79/finetunning-xlm-roberta/xlm_roberta_kg_mlm
  Vocabulary length: 250,002
  Mask token: <mask>
  Mask token ID: 250001
  PAD token: <pad>
  PAD token ID: 1

Complete vocabulary mappings identical: True

Tokenizer compatibility check passed successfully.


In [7]:
# ============================================================
# Cell 7: Convert masks and retokenize gold answers for XLM-R
# ============================================================

def tokenize_gold_answer(
    answer: str,
    tokenizer,
) -> Tuple[List[int], List[str]]:
    """
    Tokenize a gold answer without adding model special tokens.
    """
    answer = str(answer).strip()

    token_ids = tokenizer.encode(
        answer,
        add_special_tokens=False,
    )

    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    return token_ids, tokens


# Convert the BERT-style mask into the native XLM-R mask token.
evaluation_df["xlmr_prompt"] = evaluation_df["prompt"].str.replace(
    "[MASK]",
    tokenizer.mask_token,
    regex=False,
)

# Verify the converted prompt.
evaluation_df["xlmr_mask_text_count"] = (
    evaluation_df["xlmr_prompt"]
    .str.count(re.escape(tokenizer.mask_token))
)

invalid_converted_mask_df = evaluation_df[
    evaluation_df["xlmr_mask_text_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["xlmr_mask_text_count"] == 1
].copy()

xlmr_gold_token_counts = []
xlmr_gold_token_ids = []
xlmr_gold_tokens = []

for answer in tqdm(
    evaluation_df["gold_answer"].tolist(),
    desc="Tokenizing gold answers with XLM-R",
):
    token_ids, tokens = tokenize_gold_answer(
        answer=answer,
        tokenizer=tokenizer,
    )

    xlmr_gold_token_counts.append(len(token_ids))
    xlmr_gold_token_ids.append(
        token_ids[0] if len(token_ids) == 1 else None
    )
    xlmr_gold_tokens.append(tokens)

evaluation_df["xlmr_gold_token_count"] = xlmr_gold_token_counts
evaluation_df["xlmr_gold_token_id"] = xlmr_gold_token_ids
evaluation_df["xlmr_gold_tokens"] = [
    json.dumps(tokens, ensure_ascii=False)
    for tokens in xlmr_gold_tokens
]

# Save rows that are not single-token according to XLM-R.
xlmr_incompatible_df = evaluation_df[
    evaluation_df["xlmr_gold_token_count"] != 1
].copy()

# Keep only XLM-R single-token answers.
evaluation_df = evaluation_df[
    evaluation_df["xlmr_gold_token_count"] == 1
].copy()

evaluation_df["xlmr_gold_token_id"] = (
    evaluation_df["xlmr_gold_token_id"]
    .astype(int)
)

# Generic column consumed by the evaluation function.
evaluation_df["verified_gold_token_id"] = (
    evaluation_df["xlmr_gold_token_id"]
)

# Verify that IDs lie inside the tokenizer vocabulary.
valid_id_mask = (
    evaluation_df["verified_gold_token_id"].ge(0)
    & evaluation_df["verified_gold_token_id"].lt(len(tokenizer))
)

invalid_gold_id_df = evaluation_df[
    ~valid_id_mask
].copy()

evaluation_df = evaluation_df[
    valid_id_mask
].reset_index(drop=True)

print("=" * 70)
print("XLM-R gold-answer compatibility report")
print("=" * 70)
print(
    f"Rows entering XLM-R tokenization: "
    f"{len(evaluation_df) + len(xlmr_incompatible_df):,}"
)
print(
    f"Single-token answers for XLM-R:   "
    f"{len(evaluation_df):,}"
)
print(
    f"Excluded multi-token/empty answers: "
    f"{len(xlmr_incompatible_df):,}"
)
print(
    f"Invalid gold token IDs:           "
    f"{len(invalid_gold_id_df):,}"
)

if len(evaluation_df) == 0:
    raise ValueError(
        "No XLM-R-compatible single-token examples remained."
    )

display(
    evaluation_df[
        [
            "fact_id",
            "subject",
            "predicate",
            "gold_answer",
            "prompt",
            "xlmr_prompt",
            "xlmr_gold_token_count",
            "xlmr_gold_token_id",
            "xlmr_gold_tokens",
        ]
    ].head(10)
)


Tokenizing gold answers with XLM-R:   0%|          | 0/1206 [00:00<?, ?it/s]

XLM-R gold-answer compatibility report
Rows entering XLM-R tokenization: 1,206
Single-token answers for XLM-R:   672
Excluded multi-token/empty answers: 534
Invalid gold token IDs:           0


,fact_id,subject,predicate,gold_answer,prompt,xlmr_prompt,xlmr_gold_token_count,xlmr_gold_token_id,xlmr_gold_tokens
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,زادگاه محمد پورستار <mask> است.,1,212633,"[""▁اردبیل""]"
1,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,محل تولد پرم چوپرا <mask> است.,1,132163,"[""▁لاهور""]"
2,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,یاروسلاو سایفرت فردی <mask> است.,1,98560,"[""▁چکی""]"
3,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,کشور محل قرارگیری سه برخوانی، <mask> است.,1,9399,"[""▁تهران""]"
4,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,ژرژ لامپن فردی <mask> است.,1,93328,"[""▁فرانسه""]"
5,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,کشکش در استان <mask> قرار دارد.,1,124578,"[""▁گیلان""]"
6,fact_000011,اردشیر کشاورز,محل تولد,کرمانشاه,زادگاه اردشیر کشاورز [MASK] است.,زادگاه اردشیر کشاورز <mask> است.,1,168953,"[""▁کرمانشاه""]"
7,fact_000017,دهشاد پایین,استان,تهران,دهشاد پایین در استان [MASK] قرار دارد.,دهشاد پایین در استان <mask> قرار دارد.,1,9399,"[""▁تهران""]"
8,fact_000019,ابوالفضل صفاری,محل تولد,یزد,محل تولد ابوالفضل صفاری [MASK] است.,محل تولد ابوالفضل صفاری <mask> است.,1,132600,"[""▁یزد""]"
9,fact_000021,تلویزیون انقلاب ملی ایران,زبان,فارسی,زبان تلویزیون انقلاب ملی ایران [MASK] است.,زبان تلویزیون انقلاب ملی ایران <mask> است.,1,28040,"[""▁فارسی""]"


In [8]:
# ============================================================
# Cell 8: Verify mask survival after tokenization/truncation
# ============================================================

def count_tokenized_masks(
    prompt: str,
    tokenizer,
    max_length: int,
) -> int:
    """Count mask IDs after tokenization and truncation."""
    encoded = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        return_attention_mask=False,
    )

    input_ids = encoded["input_ids"]

    return int(
        sum(
            token_id == tokenizer.mask_token_id
            for token_id in input_ids
        )
    )


evaluation_df["tokenized_mask_count"] = [
    count_tokenized_masks(
        prompt=prompt,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
    for prompt in tqdm(
        evaluation_df["xlmr_prompt"].tolist(),
        desc="Checking tokenized masks",
    )
]

truncation_or_mask_problem_df = evaluation_df[
    evaluation_df["tokenized_mask_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["tokenized_mask_count"] == 1
].reset_index(drop=True)

print(
    "Rows removed because the tokenized input did not contain "
    f"exactly one mask: {len(truncation_or_mask_problem_df):,}"
)
print(
    f"Final number of evaluation examples: {len(evaluation_df):,}"
)

if len(evaluation_df) == 0:
    raise ValueError(
        "No valid examples remained after tokenized-mask validation."
    )


Checking tokenized masks:   0%|          | 0/672 [00:00<?, ?it/s]

Rows removed because the tokenized input did not contain exactly one mask: 0
Final number of evaluation examples: 672


In [9]:
# ============================================================
# Cell 9: Save the model-specific evaluation dataset
# ============================================================

evaluation_dataset_path = (
    OUTPUT_DIR / "xlmr_factual_probe_evaluation_dataset.csv"
)

xlmr_incompatible_path = (
    OUTPUT_DIR / "xlmr_excluded_non_single_token_answers.csv"
)

mask_problem_path = (
    OUTPUT_DIR / "xlmr_excluded_mask_or_truncation_problems.csv"
)

evaluation_df.to_csv(
    evaluation_dataset_path,
    index=False,
    encoding="utf-8-sig",
)

xlmr_incompatible_df.to_csv(
    xlmr_incompatible_path,
    index=False,
    encoding="utf-8-sig",
)

truncation_or_mask_problem_df.to_csv(
    mask_problem_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:")
print(f"  {evaluation_dataset_path}")
print(f"  {xlmr_incompatible_path}")
print(f"  {mask_problem_path}")


Saved:
  /kaggle/working/factual_probe_xlm_roberta/xlmr_factual_probe_evaluation_dataset.csv
  /kaggle/working/factual_probe_xlm_roberta/xlmr_excluded_non_single_token_answers.csv
  /kaggle/working/factual_probe_xlm_roberta/xlmr_excluded_mask_or_truncation_problems.csv


In [10]:
# ============================================================
# Cell 10: Token cleaning function
# ============================================================

def clean_decoded_token(
    tokenizer,
    token_id: int,
) -> str:
    """
    Convert a token ID into a readable text representation.

    XLM-R uses SentencePiece tokens, in which the '▁' symbol usually
    marks a word boundary. tokenizer.decode is used as the primary
    human-readable representation.
    """
    token_id = int(token_id)

    raw_token = tokenizer.convert_ids_to_tokens(token_id)

    decoded_token = tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    if decoded_token is None:
        decoded_token = ""

    decoded_token = str(decoded_token).strip()

    # Fall back to a normalized raw token if decoding yields an empty string.
    if decoded_token == "":
        decoded_token = str(raw_token).replace("▁", " ").strip()

    return decoded_token


In [13]:
# ============================================================
# Cell 11: Complete MLM evaluation function
# ============================================================

def evaluate_mlm_model(
    model,
    tokenizer,
    dataframe: pd.DataFrame,
    model_name: str,
    batch_size: int = 32,
    max_length: int = 128,
    top_k_to_save: int = 10,
    device: torch.device = DEVICE,
    use_amp: bool = USE_AMP,
) -> Tuple[pd.DataFrame, Dict]:
    """
    Evaluate a masked language model on a factual cloze dataset.

    Required dataframe columns:
        - xlmr_prompt
        - gold_answer
        - verified_gold_token_id

    The function calculates:
        - Full-vocabulary rank of the gold token
        - Gold-token probability and log-probability
        - P@1, P@5, and P@10 flags
        - Reciprocal rank
        - Top-k predicted token IDs, tokens, texts, and probabilities
    """

    required_columns = [
        "xlmr_prompt",
        "gold_answer",
        "verified_gold_token_id",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing evaluation columns: {missing_columns}"
        )

    if tokenizer.mask_token_id is None:
        raise ValueError(
            "The tokenizer does not define a mask token ID."
        )

    if len(dataframe) == 0:
        raise ValueError(
            "The evaluation dataframe is empty."
        )

    model = model.to(device)
    model.eval()

    result_records = []

    start_time = time.time()
    total_examples = len(dataframe)

    # Use positional indices independent of the original dataframe index.
    working_df = dataframe.reset_index(drop=True).copy()

    progress_bar = tqdm(
        range(0, total_examples, batch_size),
        desc=f"Evaluating {model_name}",
    )

    for batch_start in progress_bar:
        batch_end = min(
            batch_start + batch_size,
            total_examples,
        )

        batch_df = working_df.iloc[
            batch_start:batch_end
        ].copy()

        prompts = batch_df["xlmr_prompt"].astype(str).tolist()

        encoded = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
            return_attention_mask=True,
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        input_ids = encoded["input_ids"]

        mask_matrix = input_ids.eq(tokenizer.mask_token_id)
        masks_per_sequence = mask_matrix.sum(dim=1)

        if not torch.all(masks_per_sequence == 1):
            bad_positions = (
                torch.where(masks_per_sequence != 1)[0]
                .detach()
                .cpu()
                .tolist()
            )

            raise ValueError(
                f"{model_name}: Some tokenized sequences do not contain "
                f"exactly one mask. Batch-relative positions: "
                f"{bad_positions}"
            )

        # Get the mask position for every item in the batch.
        mask_positions = mask_matrix.long().argmax(dim=1)

        with torch.inference_mode():
            if use_amp and device.type == "cuda":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    outputs = model(**encoded)
            else:
                outputs = model(**encoded)

        logits = outputs.logits

        batch_indices = torch.arange(
            logits.size(0),
            device=device,
        )

        # Shape: [batch_size, vocabulary_size]
        mask_logits = logits[
            batch_indices,
            mask_positions,
            :,
        ].float()

        gold_token_ids = torch.tensor(
            batch_df["verified_gold_token_id"]
            .astype(int)
            .tolist(),
            dtype=torch.long,
            device=device,
        )

        if torch.any(gold_token_ids < 0):
            raise ValueError(
                f"{model_name}: A negative gold token ID was found."
            )

        if torch.any(gold_token_ids >= mask_logits.size(-1)):
            invalid_ids = gold_token_ids[
                gold_token_ids >= mask_logits.size(-1)
            ].detach().cpu().tolist()

            raise ValueError(
                f"{model_name}: Gold token IDs outside model vocabulary: "
                f"{invalid_ids}"
            )

        log_probabilities = torch.log_softmax(
            mask_logits,
            dim=-1,
        )

        gold_logits = mask_logits.gather(
            dim=1,
            index=gold_token_ids.unsqueeze(1),
        ).squeeze(1)

        gold_log_probabilities = log_probabilities.gather(
            dim=1,
            index=gold_token_ids.unsqueeze(1),
        ).squeeze(1)

        gold_probabilities = torch.exp(
            gold_log_probabilities
        )

        # Full-vocabulary rank:
        # rank = 1 + number of tokens with a strictly larger logit.
        gold_ranks = (
            mask_logits.gt(gold_logits.unsqueeze(1))
            .sum(dim=1)
            .add(1)
        )

        actual_top_k = min(
            top_k_to_save,
            mask_logits.size(-1),
        )

        top_logits, top_token_ids = torch.topk(
            mask_logits,
            k=actual_top_k,
            dim=-1,
        )

        top_log_probabilities = torch.log_softmax(
            mask_logits,
            dim=-1,
        ).gather(
            dim=1,
            index=top_token_ids,
        )

        top_probabilities = torch.exp(
            top_log_probabilities
        )

        gold_ranks_cpu = gold_ranks.detach().cpu().tolist()
        gold_probabilities_cpu = (
            gold_probabilities.detach().cpu().tolist()
        )
        gold_log_probabilities_cpu = (
            gold_log_probabilities.detach().cpu().tolist()
        )
        gold_logits_cpu = gold_logits.detach().cpu().tolist()
        top_token_ids_cpu = (
            top_token_ids.detach().cpu().tolist()
        )
        top_probabilities_cpu = (
            top_probabilities.detach().cpu().tolist()
        )

        for local_index in range(len(batch_df)):
            source_row = batch_df.iloc[local_index]

            rank = int(gold_ranks_cpu[local_index])
            gold_id = int(gold_token_ids[local_index].item())

            predicted_ids = [
                int(token_id)
                for token_id in top_token_ids_cpu[local_index]
            ]

            predicted_raw_tokens = (
                tokenizer.convert_ids_to_tokens(predicted_ids)
            )

            predicted_texts = [
                clean_decoded_token(
                    tokenizer=tokenizer,
                    token_id=token_id,
                )
                for token_id in predicted_ids
            ]

            predicted_probabilities = [
                float(probability)
                for probability in top_probabilities_cpu[local_index]
            ]

            top_predictions = []

            for prediction_rank, (
                token_id,
                raw_token,
                decoded_text,
                probability,
            ) in enumerate(
                zip(
                    predicted_ids,
                    predicted_raw_tokens,
                    predicted_texts,
                    predicted_probabilities,
                ),
                start=1,
            ):
                top_predictions.append(
                    {
                        "rank": prediction_rank,
                        "token_id": token_id,
                        "token": raw_token,
                        "text": decoded_text,
                        "probability": probability,
                    }
                )

            result_record = source_row.to_dict()

            result_record.update(
                {
                    "model_name": model_name,
                    "evaluated_prompt": source_row["xlmr_prompt"],
                    "gold_token_id_evaluated": gold_id,
                    "gold_raw_token": (
                        tokenizer.convert_ids_to_tokens(gold_id)
                    ),
                    "gold_decoded_token": clean_decoded_token(
                        tokenizer=tokenizer,
                        token_id=gold_id,
                    ),
                    "gold_logit": float(
                        gold_logits_cpu[local_index]
                    ),
                    "gold_probability": float(
                        gold_probabilities_cpu[local_index]
                    ),
                    "gold_log_probability": float(
                        gold_log_probabilities_cpu[local_index]
                    ),
                    "gold_rank": rank,
                    "reciprocal_rank": 1.0 / rank,
                    "correct_at_1": int(rank <= 1),
                    "correct_at_5": int(rank <= 5),
                    "correct_at_10": int(rank <= 10),
                    "top1_token_id": predicted_ids[0],
                    "top1_raw_token": predicted_raw_tokens[0],
                    "top1_text": predicted_texts[0],
                    "top1_probability": predicted_probabilities[0],
                    "top_token_ids": json.dumps(
                        predicted_ids,
                        ensure_ascii=False,
                    ),
                    "top_raw_tokens": json.dumps(
                        predicted_raw_tokens,
                        ensure_ascii=False,
                    ),
                    "top_decoded_tokens": json.dumps(
                        predicted_texts,
                        ensure_ascii=False,
                    ),
                    "top_probabilities": json.dumps(
                        predicted_probabilities,
                        ensure_ascii=False,
                    ),
                    "top_predictions": json.dumps(
                        top_predictions,
                        ensure_ascii=False,
                    ),
                }
            )

            result_records.append(result_record)

        processed_examples = batch_end
        elapsed = time.time() - start_time

        progress_bar.set_postfix(
            processed=processed_examples,
            examples_per_second=(
                processed_examples / elapsed
                if elapsed > 0
                else 0.0
            ),
        )

        # Explicitly release batch tensors.
        del encoded
        del input_ids
        del mask_matrix
        del logits
        del mask_logits
        del log_probabilities
        del outputs

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed_time = time.time() - start_time

    result_df = pd.DataFrame(result_records)

    speed_info = {
        "model_name": model_name,
        "samples": total_examples,
        "elapsed_seconds": elapsed_time,
        "samples_per_second": (
            total_examples / elapsed_time
            if elapsed_time > 0
            else float("nan")
        ),
        "batch_size": batch_size,
        "max_length": max_length,
        "device": str(device),
        "mixed_precision": bool(use_amp and device.type == "cuda"),
    }

    return result_df, speed_info


In [14]:
# ============================================================
# Cell 12: Metric calculation functions
# ============================================================

def calculate_metrics(
    result_df: pd.DataFrame,
    model_name: str,
) -> Dict:
    """Calculate aggregate factual probing metrics."""

    if len(result_df) == 0:
        raise ValueError(
            f"The result dataframe for {model_name} is empty."
        )

    required_columns = [
        "gold_rank",
        "reciprocal_rank",
        "correct_at_1",
        "correct_at_5",
        "correct_at_10",
        "gold_probability",
        "gold_log_probability",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in result_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing metric columns: {missing_columns}"
        )

    ranks = pd.to_numeric(
        result_df["gold_rank"],
        errors="raise",
    )

    return {
        "model": model_name,
        "samples": int(len(result_df)),
        "P@1": float(result_df["correct_at_1"].mean()),
        "P@5": float(result_df["correct_at_5"].mean()),
        "P@10": float(result_df["correct_at_10"].mean()),
        "MRR": float(result_df["reciprocal_rank"].mean()),
        "Mean_Rank": float(ranks.mean()),
        "Median_Rank": float(ranks.median()),
        "Mean_Gold_Probability": float(
            result_df["gold_probability"].mean()
        ),
        "Mean_Gold_LogProbability": float(
            result_df["gold_log_probability"].mean()
        ),
    }


def calculate_grouped_metrics(
    result_df: pd.DataFrame,
    model_name: str,
    group_column: str = "predicate",
) -> pd.DataFrame:
    """Calculate factual probing metrics for each predicate/group."""

    if group_column not in result_df.columns:
        raise ValueError(
            f"Grouping column '{group_column}' was not found."
        )

    grouped_records = []

    for group_value, group_df in result_df.groupby(
        group_column,
        dropna=False,
    ):
        metrics = calculate_metrics(
            result_df=group_df,
            model_name=model_name,
        )

        metrics[group_column] = group_value
        grouped_records.append(metrics)

    grouped_metrics_df = pd.DataFrame(grouped_records)

    preferred_columns = [
        "model",
        group_column,
        "samples",
        "P@1",
        "P@5",
        "P@10",
        "MRR",
        "Mean_Rank",
        "Median_Rank",
        "Mean_Gold_Probability",
        "Mean_Gold_LogProbability",
    ]

    return grouped_metrics_df[preferred_columns]


In [15]:
# ============================================================
# Cell 13: Load and evaluate baseline XLM-R
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 70)
print("Loading baseline XLM-R")
print("=" * 70)

xlmr_base_model = AutoModelForMaskedLM.from_pretrained(
    XLMR_BASE_MODEL,
)

print(f"Model type: {xlmr_base_model.config.model_type}")
print(f"Model vocabulary size: {xlmr_base_model.config.vocab_size}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")

if xlmr_base_model.config.vocab_size != len(tokenizer):
    raise ValueError(
        "Baseline model and tokenizer vocabulary sizes differ:\n"
        f"Model: {xlmr_base_model.config.vocab_size}\n"
        f"Tokenizer: {len(tokenizer)}"
    )

xlmr_base_model.to(DEVICE)
xlmr_base_model.eval()

baseline_results_xlmr, baseline_speed_xlmr = evaluate_mlm_model(
    model=xlmr_base_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="XLM-R baseline",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
    device=DEVICE,
    use_amp=USE_AMP,
)

print("\nBaseline evaluation completed.")
print(json.dumps(
    baseline_speed_xlmr,
    indent=2,
    ensure_ascii=False,
))

baseline_output_path = (
    OUTPUT_DIR / "xlmr_baseline_factual_probe_results.csv"
)

baseline_results_xlmr.to_csv(
    baseline_output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nBaseline row-level results saved to:\n{baseline_output_path}")

display(
    baseline_results_xlmr[
        [
            "fact_id",
            "subject",
            "predicate",
            "gold_answer",
            "gold_rank",
            "gold_probability",
            "top1_text",
            "top1_probability",
            "top_decoded_tokens",
        ]
    ].head(10)
)


Loading baseline XLM-R


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model type: xlm-roberta
Model vocabulary size: 250002
Tokenizer vocabulary size: 250002


Evaluating XLM-R baseline:   0%|          | 0/21 [00:00<?, ?it/s]


Baseline evaluation completed.
{
  "model_name": "XLM-R baseline",
  "samples": 672,
  "elapsed_seconds": 54.821309328079224,
  "samples_per_second": 12.25800711869909,
  "batch_size": 32,
  "max_length": 128,
  "device": "cpu",
  "mixed_precision": false
}

Baseline row-level results saved to:
/kaggle/working/factual_probe_xlm_roberta/xlmr_baseline_factual_probe_results.csv


,fact_id,subject,predicate,gold_answer,gold_rank,gold_probability,top1_text,top1_probability,top_decoded_tokens
0,fact_000001,محمد پورستار,محل تولد,اردبیل,69,0.001362,ی,0.164677,"[""ی"", ""ه"", ""آباد"", ""بوده"", ""کی"", ""ک"", ""ایی"", ""ایران"", ""تهران"", ""مشهد""]"
1,fact_000003,پرم چوپرا,محل تولد,لاهور,167,0.000965,بوده,0.123714,"[""بوده"", ""پور"", ""گو"", ""وس"", ""ستان"", ""سر"", ""آباد"", ""لام"", ""گون"", ""چی""]"
2,fact_000004,یاروسلاو سایفرت,ملیت,چکی,884,0.000083,موجود,0.049012,"[""موجود"", ""فعال"", ""شده"", ""مشهور"", ""بوده"", ""سالم"", ""معتبر"", ""واحد"", ""معروف"", ""جدید""]"
3,fact_000005,سه برخوانی,کشور,تهران,5,0.027363,ایران,0.116407,"[""ایران"", ""افغانستان"", ""بوده"", ""پاکستان"", ""تهران"", ""ارومیه"", ""عراق"", ""قزوین"", ""هندوستان"", ""هند""]"
4,fact_000009,ژرژ لامپن,ملیت,فرانسه,918,0.000132,ت,0.046119,"[""ت"", ""شده"", ""بوده"", ""داشته"", ""مشهور"", ""ن"", ""."", ""سالم"", ""موجود"", ""س""]"
5,fact_000010,کشکش,استان,گیلان,15,0.023808,کردستان,0.096836,"[""کردستان"", ""مازندران"", ""هرمزگان"", ""فارس"", ""گلستان"", ""کرمانشاه"", ""خوزستان"", ""لرستان"", ""مرکزی"", ""اردبیل""]"
6,fact_000011,اردشیر کشاورز,محل تولد,کرمانشاه,32,0.002838,بوده,0.452948,"[""بوده"", ""شده"", ""ایرانی"", ""ایران"", ""اصفهان"", ""بختیاری"", ""معروف"", ""کرمان"", ""روستایی"", ""شیراز""]"
7,fact_000017,دهشاد پایین,استان,تهران,18,0.011807,مازندران,0.161586,"[""مازندران"", ""لرستان"", ""گلستان"", ""کرمانشاه"", ""خوزستان"", ""هرمزگان"", ""فارس"", ""بوشهر"", ""اصفهان"", ""گیلان""]"
8,fact_000019,ابوالفضل صفاری,محل تولد,یزد,34,0.004984,تهران,0.094672,"[""تهران"", ""ه"", ""بوده"", ""مشهد"", ""ان"", ""همدان"", ""اهواز"", ""قم"", ""اصفهان"", ""شیراز""]"
9,fact_000021,تلویزیون انقلاب ملی ایران,زبان,فارسی,1,0.095042,فارسی,0.095042,"[""فارسی"", ""اسلامی"", ""بوده"", ""افغانستان"", ""عربی"", ""ایران"", ""زنده"", ""</s>"", ""زبان"", ""این""]"


In [16]:
# ============================================================
# Cell 14: Release the baseline model from GPU memory
# ============================================================

del xlmr_base_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Baseline model was released from memory.")


Baseline model was released from memory.


In [17]:
# ============================================================
# Cell 15: Load and evaluate fine-tuned XLM-R-KG
# ============================================================

print("=" * 70)
print("Loading XLM-R-KG")
print("=" * 70)
print(f"Model path: {XLMR_KG_MODEL_PATH}")

xlmr_kg_model = AutoModelForMaskedLM.from_pretrained(
    str(XLMR_KG_MODEL_PATH),
)

print(f"Model type: {xlmr_kg_model.config.model_type}")
print(f"Model vocabulary size: {xlmr_kg_model.config.vocab_size}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")

if xlmr_kg_model.config.vocab_size != len(tokenizer):
    raise ValueError(
        "KG model and tokenizer vocabulary sizes differ:\n"
        f"Model: {xlmr_kg_model.config.vocab_size}\n"
        f"Tokenizer: {len(tokenizer)}"
    )

xlmr_kg_model.to(DEVICE)
xlmr_kg_model.eval()

kg_results_xlmr, kg_speed_xlmr = evaluate_mlm_model(
    model=xlmr_kg_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="XLM-R KG",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
    device=DEVICE,
    use_amp=USE_AMP,
)

print("\nXLM-R-KG evaluation completed.")
print(json.dumps(
    kg_speed_xlmr,
    indent=2,
    ensure_ascii=False,
))

kg_output_path = (
    OUTPUT_DIR / "xlmr_kg_factual_probe_results.csv"
)

kg_results_xlmr.to_csv(
    kg_output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\nKG row-level results saved to:\n{kg_output_path}")

display(
    kg_results_xlmr[
        [
            "fact_id",
            "subject",
            "predicate",
            "gold_answer",
            "gold_rank",
            "gold_probability",
            "top1_text",
            "top1_probability",
            "top_decoded_tokens",
        ]
    ].head(10)
)


Loading XLM-R-KG
Model path: /kaggle/input/notebooks/sammir79/finetunning-xlm-roberta/xlm_roberta_kg_mlm


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model type: xlm-roberta
Model vocabulary size: 250002
Tokenizer vocabulary size: 250002


Evaluating XLM-R KG:   0%|          | 0/21 [00:00<?, ?it/s]


XLM-R-KG evaluation completed.
{
  "model_name": "XLM-R KG",
  "samples": 672,
  "elapsed_seconds": 62.47063207626343,
  "samples_per_second": 10.757054597744908,
  "batch_size": 32,
  "max_length": 128,
  "device": "cpu",
  "mixed_precision": false
}

KG row-level results saved to:
/kaggle/working/factual_probe_xlm_roberta/xlmr_kg_factual_probe_results.csv


,fact_id,subject,predicate,gold_answer,gold_rank,gold_probability,top1_text,top1_probability,top_decoded_tokens
0,fact_000001,محمد پورستار,محل تولد,اردبیل,30,1.924467e-03,تهران,0.330727,"[""تهران"", ""ایران"", ""اصفهان"", ""گیلان"", ""ایرانی"", ""مشهد"", ""شیراز"", ""بوشهر"", ""رشت"", ""ه""]"
1,fact_000003,پرم چوپرا,محل تولد,لاهور,283,1.620353e-04,،,0.542516,"[""،"", ""ژاپن"", ""ایران"", ""مسکو"", ""ایتالیا"", ""تهران"", ""لندن"", ""کانادا"", ""هند"", ""چین""]"
2,fact_000004,یاروسلاو سایفرت,ملیت,چکی,2200,4.921263e-07,فرانسوی,0.461920,"[""فرانسوی"", ""شخصیت"", ""آمریکایی"", ""ایرانی"", ""روسی"", ""ن"", ""بازیگر"", ""انگلیسی"", ""روس"", ""فعال""]"
3,fact_000005,سه برخوانی,کشور,تهران,42,5.017688e-04,ایران,0.631060,"[""ایران"", ""آمریکا"", ""بریتانیا"", ""فرانسه"", ""آلمان"", ""افغانستان"", ""عراق"", ""ایتالیا"", ""انگلستان"", ""اسپانیا""]"
4,fact_000009,ژرژ لامپن,ملیت,فرانسه,8,1.592339e-03,فرانسوی,0.859538,"[""فرانسوی"", ""شخصیت"", ""ایرانی"", ""آمریکایی"", ""بازیگر"", ""نویسنده"", ""ن"", ""فرانسه"", ""انگلیسی"", ""شاعر""]"
5,fact_000010,کشکش,استان,گیلان,2,2.300126e-01,مازندران,0.267124,"[""مازندران"", ""گیلان"", ""خوزستان"", ""کرمانشاه"", ""گلستان"", ""هرمزگان"", ""مرکزی"", ""کردستان"", ""اردبیل"", ""فارس""]"
6,fact_000011,اردشیر کشاورز,محل تولد,کرمانشاه,11,1.339242e-02,تهران,0.377096,"[""تهران"", ""ایران"", ""مشهد"", ""اصفهان"", ""شیراز"", ""رشت"", ""ایرانی"", ""کرمان"", ""گیلان"", ""همدان""]"
7,fact_000017,دهشاد پایین,استان,تهران,20,4.077444e-03,گیلان,0.248534,"[""گیلان"", ""مازندران"", ""کرمانشاه"", ""مرکزی"", ""گلستان"", ""خوزستان"", ""اردبیل"", ""فارس"", ""بوشهر"", ""هرمزگان""]"
8,fact_000019,ابوالفضل صفاری,محل تولد,یزد,8,1.119885e-02,تهران,0.578001,"[""تهران"", ""اصفهان"", ""ایران"", ""تبریز"", ""شیراز"", ""قم"", ""مشهد"", ""یزد"", ""،"", ""بوشهر""]"
9,fact_000021,تلویزیون انقلاب ملی ایران,زبان,فارسی,1,8.232131e-01,فارسی,0.823213,"[""فارسی"", ""انگلیسی"", ""عربی"", ""فرانسوی"", ""ایرانی"", ""روسی"", ""کردی"", ""ترکی"", ""،"", ""زبان""]"


In [18]:
# ============================================================
# Cell 16: Calculate overall metrics
# ============================================================

baseline_metrics = calculate_metrics(
    result_df=baseline_results_xlmr,
    model_name="XLM-R baseline",
)

kg_metrics = calculate_metrics(
    result_df=kg_results_xlmr,
    model_name="XLM-R KG",
)

xlmr_metrics_df = pd.DataFrame(
    [
        baseline_metrics,
        kg_metrics,
    ]
)

print("=" * 70)
print("Overall factual probing metrics")
print("=" * 70)

display(xlmr_metrics_df.round(6))

xlmr_metrics_percent_df = xlmr_metrics_df.copy()

for column in ["P@1", "P@5", "P@10"]:
    xlmr_metrics_percent_df[column] = (
        xlmr_metrics_percent_df[column] * 100
    )

print("P@K values shown as percentages:")
display(xlmr_metrics_percent_df.round(4))


Overall factual probing metrics


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,XLM-R baseline,672,0.105655,0.313988,0.447917,0.214922,222.373512,12.5,0.055568,-4.662091
1,XLM-R KG,672,0.114583,0.352679,0.547619,0.240559,123.629464,9.0,0.090119,-4.606514


P@K values shown as percentages:


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,XLM-R baseline,672,10.5655,31.3988,44.7917,0.2149,222.3735,12.5,0.0556,-4.6621
1,XLM-R KG,672,11.4583,35.2679,54.7619,0.2406,123.6295,9.0,0.0901,-4.6065
